![](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# Cap 03 — Herramientas y ReAct

profesor [Carlos Daniel Jiménez](danieljimenez88m@gmail.com)


Añadimos herramientas al agente:
- `@tool`: decorador para crear herramientas desde funciones Python
- `ToolNode`: nodo especial que ejecuta herramientas
- `tools_condition`: arista condicional para el loop ReAct
- Loop: LLM → [¿tool_call?] → Tools → LLM → END

**Dominio**: Álbumes de Miles Davis  
**Flujo**: `START → razonador → [tools_condition] → tools → razonador → END`

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Falta OPENAI_API_KEY en .env"

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from rich import print as rprint

# Cargar datos
DATA_PATH = Path("../00_datos/davis_albums.json")
with open(DATA_PATH, encoding="utf-8") as f:
    davis_albums = json.load(f)

print(f"Cargados {len(davis_albums)} álbumes de Davis")

## 1. Definir Herramientas con `@tool`

El decorador `@tool` convierte una función Python en una herramienta que el LLM puede llamar.  
LangChain genera automáticamente el JSON Schema para el LLM a partir del docstring y los tipos.

In [ ]:
@tool
def buscar_album(titulo: str) -> str:
    """Busca información sobre un álbum específico de Miles Davis por título.
    
    Args:
        titulo: Título del álbum a buscar
    
    Returns:
        Descripción del álbum o mensaje de error si no se encuentra
    """
    titulo_lower = titulo.lower()
    for album in davis_albums:
        if titulo_lower in album["titulo"].lower():
            return json.dumps({
                "titulo": album["titulo"],
                "año": album["año"],
                "epocas": album["epocas"],
                "descripcion": album["descripcion"],
                "importancia": album["importancia_historica"]
            }, ensure_ascii=False)
    return f"No se encontró el álbum '{titulo}'. Álbumes disponibles: {[a['titulo'] for a in davis_albums]}"

@tool
def listar_epocas() -> str:
    """Lista todas las épocas/períodos musicales en la discografía de Miles Davis.

    Returns:
        Lista JSON con los nombres de todas las épocas ordenadas alfabéticamente
    """
    epocas = set()
    for album in davis_albums:
        epocas.update(album.get("epocas", []))
    return json.dumps(sorted(list(epocas)), ensure_ascii=False)

@tool
def get_influencias(album_titulo: str) -> str:
    """Obtiene las influencias musicales de un álbum de Miles Davis.
    
    Args:
        album_titulo: Título del álbum
    
    Returns:
        Influencias que recibió y artistas que influyó
    """
    for album in davis_albums:
        if album_titulo.lower() in album["titulo"].lower():
            return json.dumps({
                "influencias_recibidas": album.get("influencias", []),
                "artistas_influenciados": album.get("influenciados", [])
            }, ensure_ascii=False)
    return f"No se encontró '{album_titulo}'"

print("Herramientas definidas ✓")
print("\nJSON Schema auto-generado para buscar_album:")
rprint(buscar_album.args_schema.model_json_schema())

## 2. El JSON Schema Auto-Generado

El decorador `@tool` inspecciona los tipos y docstring para generar el schema que el LLM usa para llamar a la herramienta.

In [ ]:
tools = [buscar_album, listar_epocas, get_influencias]

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), temperature=0)

# bind_tools informa al LLM que puede usar estas herramientas
llm_with_tools = llm.bind_tools(tools)

print(f"LLM con {len(tools)} herramientas:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

## 3. Construir el Loop ReAct

El patrón ReAct (Reasoning + Acting):
1. LLM razona sobre qué herramienta usar
2. Si decide usar una: `ToolNode` la ejecuta
3. El resultado vuelve al LLM
4. El LLM decide si necesita más herramientas o puede responder

In [ ]:
DAVIS_SYSTEM = """Eres un musicólogo especializado en Miles Davis.
Usa las herramientas disponibles para responder preguntas sobre sus álbumes.
Siempre consulta los datos antes de responder para ser preciso."""

def node_razonador(state: MessagesState) -> dict:
    """El LLM decide qué herramienta usar (o responde directamente)."""
    response = llm_with_tools.invoke(
        [SystemMessage(content=DAVIS_SYSTEM)] + state["messages"]
    )
    return {"messages": [response]}

# ToolNode ejecuta automáticamente las llamadas a herramientas
tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("razonador", node_razonador)
builder.add_node("tools", tool_node)

builder.add_edge(START, "razonador")
# tools_condition: si el LLM hizo tool_calls → "tools", si no → END
builder.add_conditional_edges("razonador", tools_condition)
builder.add_edge("tools", "razonador")  # El loop vuelve al LLM

graph = builder.compile()
print("Grafo ReAct compilado ✓")

## 4. Visualizar el Loop ReAct

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
result = graph.invoke({
    "messages": [HumanMessage(content="¿Qué técnicas musicales innovó Miles Davis en Kind of Blue?")]
})

print("=== Respuesta Final ===")
print(result["messages"][-1].content)
print(f"\nTotal mensajes: {len(result['messages'])}")

In [ ]:
# Ver el trace completo: Human → AI(tool_call) → Tool → AI(respuesta)
print("=== Trace Completo ===")
for msg in result["messages"]:
    tipo = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"[{tipo}] Tool calls: {[tc['name'] for tc in msg.tool_calls]}")
    elif hasattr(msg, "name") and msg.name:
        print(f"[ToolMessage:{msg.name}] {str(msg.content)[:100]}...")
    else:
        print(f"[{tipo}] {str(msg.content)[:100]}...")

## 5. Avanzado: Formateador Post-Loop

Añadimos un nodo final que convierte el último AIMessage en respuesta formateada.

In [ ]:
def node_formateador_final(state: MessagesState) -> dict:
    """Formatea la respuesta final en prosa estructurada."""
    ultimo_ai = next(
        (m for m in reversed(state["messages"]) if isinstance(m, AIMessage)),
        None
    )
    if not ultimo_ai:
        return {}
    
    formateado = f"""
╔══════════════════════════════════════╗
║  ANÁLISIS MUSICAL — MILES DAVIS      ║
╚══════════════════════════════════════╝

{ultimo_ai.content}

Fuente: Base de datos discográfica Miles Davis
"""
    return {"messages": [AIMessage(content=formateado)]}

# Nuevo grafo con formateador
builder2 = StateGraph(MessagesState)
builder2.add_node("razonador", node_razonador)
builder2.add_node("tools", tool_node)
builder2.add_node("formateador", node_formateador_final)

builder2.add_edge(START, "razonador")
builder2.add_conditional_edges("razonador", tools_condition, 
                               {"tools": "tools", END: "formateador"})
builder2.add_edge("tools", "razonador")
builder2.add_edge("formateador", END)

graph2 = builder2.compile()

result2 = graph2.invoke({
    "messages": [HumanMessage(content="¿Cuáles son las épocas de Davis y cuál es la más revolucionaria?")]
})
print(result2["messages"][-1].content)

## Resumen del Capítulo

| Concepto | Descripción |
|---------|-------------|
| `@tool` | Convierte función en herramienta con JSON Schema automático |
| `bind_tools()` | Informa al LLM qué herramientas puede usar |
| `ToolNode` | Nodo que ejecuta las herramientas solicitadas por el LLM |
| `tools_condition` | Arista condicional: tool_call → tools, no tool_call → END |
| Loop ReAct | razonador ↔ tools hasta que el LLM decide responder |

**Próximo capítulo**: Enrutamiento condicional con múltiples dominios